# Detecção de fraude em cartão de crédito

Notebook em 5 etapas: **Explorar → Preparar → Treinar → Avaliar → Explicar**.

Dataset: Credit Card Fraud Detection (ULB/Kaggle). Colunas `Time`, `Amount`, `Class` e `V1`–`V28` (já transformadas por PCA). Só ~0,17% das transações são fraude, e esse desbalanceamento guia todas as decisões abaixo.

Regra de ouro do notebook: tudo que "aprende" com os dados (scaler, SMOTE, undersampling, limiar) é ajustado **só no treino/validação**. O teste é usado uma única vez, no fim.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier
import shap

## Etapa 1 - Explorar

**Objetivo:** entender o dado antes de modelar: tamanho do desbalanceamento, nulos, duplicatas e o comportamento de `Amount` e `Time`. Aqui só olhamos; nada é ajustado.

In [ ]:
URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
df = pd.read_csv(URL)

print(df.shape)  # esperado: (284807, 31)
df.info()        # todas as colunas devem ser numéricas
df.describe().T  # V1..V28 vêm de PCA (média ~0); só Time, Amount e Class são interpretáveis

### Qualidade dos dados

Duplicatas saem **antes** da divisão treino/teste. Se uma cópia cair em cada lado, o modelo "viu" o teste e a métrica fica inflada.

In [ ]:
print("Nulos:", df.isna().sum().sum())
print("Duplicadas:", df.duplicated().sum())

# keep="first" mantém a primeira ocorrência; reset_index evita buracos no índice
df = df.drop_duplicates().reset_index(drop=True)
print("Shape após remover duplicatas:", df.shape)

### Desbalanceamento

Um modelo "burro" que diz sempre `0` (não fraude) acerta ~99,8%. Acurácia, portanto, não serve como métrica aqui.

In [ ]:
print(df["Class"].value_counts())
print(df["Class"].value_counts(normalize=True) * 100)

acuracia_burra = (df["Class"] == 0).mean()
print(f"Acurácia de prever sempre 0: {acuracia_burra:.4%}")

# escala log, senão a barra da fraude some
df["Class"].value_counts().plot(kind="bar", logy=True, title="Classes (escala log)")
plt.savefig("classes.png", dpi=120, bbox_inches="tight")
plt.show()

### `Amount`

Muito assimétrico (muitos valores baixos, poucos altos). **Não** remova outliers: em fraude, valores extremos podem ser o sinal.

In [ ]:
print(df.groupby("Class")["Amount"].describe())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for classe, nome in [(0, "Normal"), (1, "Fraude")]:
    ax[classe].hist(np.log1p(df.loc[df["Class"] == classe, "Amount"]), bins=50)
    ax[classe].set_title(f"log1p(Amount) - {nome}")
plt.savefig("amount_log.png", dpi=120, bbox_inches="tight")
plt.show()

### `Time`

É um contador de **segundos** desde a primeira transação (~2 dias), não um horário do dia. Para ver padrão por hora, convertemos. Poucas fraudes por hora deixam a taxa ruidosa; use para investigar, não para concluir.

In [ ]:
hora_do_dia = (df["Time"] // 3600) % 24
df.groupby(hora_do_dia)["Class"].agg(["count", "sum", "mean"])

### Minhas conclusões da exploração

*(escreva aqui, com suas palavras: o que você viu e o que muda nas próximas etapas - divisão dos dados, métrica, tratamento de `Amount` e `Time`)*

## Etapa 2 - Preparar

**Objetivo:** dividir os dados e montar o pré-processamento **sem vazamento** (data leakage).

- `stratify=y` mantém a mesma proporção de fraude (~0,17%) em cada parte. Sem isso, o acaso pode deixar o teste com poucas fraudes (ou nenhuma).
- Três conjuntos: **treino** (aprende), **validação** (escolhe modelo e limiar), **teste** (avaliação final, uma vez).

In [ ]:
X = df.drop(columns="Class")
y = df["Class"]

# 20% para teste; do restante, 25% (= 20% do total) para validação
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

# a taxa de fraude deve ser quase igual nos três conjuntos
for nome, alvo in [("treino", y_train), ("validação", y_val), ("teste", y_test)]:
    print(f"{nome:10s} linhas={len(alvo):6d}  fraudes={alvo.sum():4d}  taxa={alvo.mean():.4%}")

### Pré-processamento (definido aqui, ajustado só no treino)

`V1`–`V28` já estão em escala parecida; escalamos só `Time` e `Amount`. Este objeto **ainda não foi ajustado**: o `fit` acontece dentro do `Pipeline`, só nos dados de treino. Na validação e no teste o Pipeline aplica apenas `transform`, com a média e o desvio do treino. Chamar `fit_transform` no dataset inteiro seria vazamento.

Decisões suas para refinar depois: `Time` é um contador (pode ser removido ou virar hora do dia) e `Amount` é assimétrico (um `np.log1p` pode ajudar).

In [ ]:
preprocessador = ColumnTransformer(
    transformers=[("escala", StandardScaler(), ["Time", "Amount"])],
    remainder="passthrough",  # V1..V28 passam sem alteração
)

## Etapa 3 - Treinar

**Objetivo:** treinar candidatos com estratégias diferentes para o desbalanceamento e compará-los **na validação**.

| Estratégia | Ideia |
|---|---|
| `class_weight` / `scale_pos_weight` | erro na fraude pesa mais; dados intactos |
| Undersampling | descarta normais no treino |
| SMOTE (oversampling) | cria fraudes sintéticas por interpolação |

Reamostragem só dentro de um `imblearn.pipeline.Pipeline`, para valer apenas no treino. Reamostrar antes do split vazaria fraudes sintéticas para validação e teste.

In [ ]:
# nº de normais / nº de fraudes no TREINO (equivale a class_weight="balanced")
peso_fraude = (y_train == 0).sum() / (y_train == 1).sum()

modelos = {
    # piso: qualquer modelo útil precisa bater isso em recall/PR-AUC
    "Baseline (sempre 0)": Pipeline([
        ("modelo", DummyClassifier(strategy="most_frequent")),
    ]),
    "LogReg sem tratamento": Pipeline([
        ("prep", clone(preprocessador)),
        ("modelo", LogisticRegression(max_iter=1000)),
    ]),
    "LogReg class_weight": Pipeline([
        ("prep", clone(preprocessador)),
        ("modelo", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]),
    # sampling_strategy=0.1 -> 10 normais por fraude depois do corte
    "LogReg undersampling": ImbPipeline([
        ("prep", clone(preprocessador)),
        ("amostra", RandomUnderSampler(sampling_strategy=0.1, random_state=42)),
        ("modelo", LogisticRegression(max_iter=1000)),
    ]),
    "LogReg SMOTE": ImbPipeline([
        ("prep", clone(preprocessador)),
        ("amostra", SMOTE(sampling_strategy=0.1, random_state=42)),
        ("modelo", LogisticRegression(max_iter=1000)),
    ]),
    "XGBoost scale_pos_weight": Pipeline([
        ("prep", clone(preprocessador)),
        ("modelo", XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.1,
            scale_pos_weight=peso_fraude, eval_metric="aucpr",
            random_state=42, n_jobs=-1,
        )),
    ]),
}

### Comparação na validação (limiar padrão 0.5)

Métricas da classe **fraude**:
- **recall** = das fraudes reais, quantas peguei;
- **precisão** = dos alertas que dei, quantos eram fraude;
- **F1** = equilíbrio entre os dois;
- **PR-AUC** = área da curva precisão-recall, que não depende do limiar.

Olhe recall **e** precisão juntos: recall alto com precisão baixa = muito cliente legítimo bloqueado.

In [ ]:
linhas = []
for nome, pipe in modelos.items():
    pipe.fit(X_train, y_train)                 # fit SÓ no treino
    pred = pipe.predict(X_val)
    proba = pipe.predict_proba(X_val)[:, 1]
    linhas.append({
        "modelo": nome,
        "recall": recall_score(y_val, pred),
        "precisao": precision_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred),
        "pr_auc": average_precision_score(y_val, proba),
    })

resultados = pd.DataFrame(linhas).set_index("modelo").round(3)
resultados.sort_values("pr_auc", ascending=False)

**Leitura:** o baseline tem recall 0 com ~99,8% de acurácia. `class_weight`, SMOTE e undersampling elevam o recall mas derrubam a precisão (o `class_weight` com regressão logística gera muito alarme falso). O XGBoost tem o melhor PR-AUC e o melhor equilíbrio. Com só ~94 fraudes na validação, diferenças pequenas entre modelos são ruído.

## Etapa 4 - Avaliar

**Objetivo:** entender o que o modelo acerta e erra, escolher o limiar de decisão **na validação** e medir o resultado final no **teste**, uma vez.

In [ ]:
modelo_final = modelos["XGBoost scale_pos_weight"]  # já ajustado no treino
proba_val = modelo_final.predict_proba(X_val)[:, 1]
proba_test = modelo_final.predict_proba(X_test)[:, 1]

### Matriz de confusão

Linhas = real, colunas = previsto:

|  | previsto normal | previsto fraude |
|---|---|---|
| **real normal** | VN | FP (alarme falso) |
| **real fraude** | FN (fraude que passou) | VP (fraude pega) |

Exemplo: com VP=80, FN=14 e FP=10, o recall é 80/(80+14) ≈ 85% e a precisão é 80/(80+10) ≈ 89%. A acurácia seria ~99,96% e não diz quase nada sobre a fraude.

In [ ]:
pred_05 = (proba_val >= 0.5).astype(int)
print("Matriz de confusão (validação, limiar 0.5):")
print(confusion_matrix(y_val, pred_05))

### ROC vs. Precisão-Recall

- **ROC** (recall x taxa de falsos positivos): com 99,8% de normais, milhares de falsos alarmes mexem pouco na taxa de FP, então a ROC parece ótima mesmo com modelo medíocre.
- **PR** (precisão x recall): ignora os verdadeiros negativos e mostra o custo real dos falsos alarmes. Em dados desbalanceados é mais informativa. A linha de referência é a taxa de fraude (~0,17%), não 0,5.

In [ ]:
print(f"ROC-AUC: {roc_auc_score(y_val, proba_val):.4f}")
print(f"PR-AUC : {average_precision_score(y_val, proba_val):.4f}")

fpr, tpr, _ = roc_curve(y_val, proba_val)
prec, rec, limiares = precision_recall_curve(y_val, proba_val)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(fpr, tpr)
ax[0].plot([0, 1], [0, 1], "--", color="gray")
ax[0].set(title="ROC", xlabel="Taxa de falsos positivos", ylabel="Recall")
ax[1].plot(rec, prec)
ax[1].axhline(y_val.mean(), ls="--", color="gray")  # referência: acaso
ax[1].set(title="Precisão-Recall", xlabel="Recall", ylabel="Precisão")
plt.tight_layout()
plt.savefig("curvas_roc_pr.png", dpi=120)
plt.show()

### Escolha do limiar (na validação, nunca no teste)

0.5 é só o padrão. Escolher o limiar no teste seria vazamento: o teste deixaria de ser dado "nunca visto".

Critério abaixo: maximizar o F1 da fraude. Em produção, decida pelo custo de negócio: se perder uma fraude custa muito mais que bloquear um cliente legítimo, prefira recall alto (por exemplo, "o maior limiar com recall ≥ 90%"). **Atenção:** com ~94 fraudes na validação, o limiar de F1 máximo é instável e tende a favorecer a precisão.

In [ ]:
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
limiar = limiares[np.argmax(f1s)]
print(f"Limiar escolhido na validação: {limiar:.3f}")

### Avaliação final no teste (uma vez, com o limiar já fixado)

In [ ]:
pred_test = (proba_test >= limiar).astype(int)
print("Matriz de confusão (teste):")
print(confusion_matrix(y_test, pred_test))
print(f"Recall   : {recall_score(y_test, pred_test):.3f}")
print(f"Precisão : {precision_score(y_test, pred_test):.3f}")
print(f"F1       : {f1_score(y_test, pred_test):.3f}")
print(f"PR-AUC   : {average_precision_score(y_test, proba_test):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, proba_test):.3f}")

Com ~95 fraudes no teste, cada fraude vale ~1 ponto de recall: trate os números como estimativa com margem de erro.

### Minhas conclusões da avaliação

*(escreva: o trade-off entre recall e precisão e qual critério de limiar faz sentido para o negócio)*

## Etapa 5 - Explicar

**Objetivo:** entender **por que** o modelo marca uma transação como fraude: visão global (quais variáveis importam) e local (por que *esta* transação).

Limite importante: `V1`–`V28` vêm de PCA e não têm significado de negócio. O SHAP diz "V14 empurrou para fraude", mas não diz o que `V14` representa. SHAP explica o **modelo**, não a causa real da fraude.

In [ ]:
prep = modelo_final.named_steps["prep"]
xgb = modelo_final.named_steps["modelo"]

# nomes como "escala__Amount" -> "Amount"
nomes = [n.split("__")[-1] for n in prep.get_feature_names_out()]
X_val_t = pd.DataFrame(prep.transform(X_val), columns=nomes, index=X_val.index)

# todas as fraudes da validação + amostra de normais (para acelerar o SHAP)
rng = np.random.RandomState(42)
idx_fraude = y_val[y_val == 1].index
idx_normal = rng.choice(y_val[y_val == 0].index, size=3000, replace=False)
amostra = X_val_t.loc[np.concatenate([idx_fraude, idx_normal])]

# importância nativa do XGBoost (rápida, mas pode enganar com variáveis correlacionadas)
pd.Series(xgb.feature_importances_, index=nomes).sort_values(ascending=False).head(10)

### SHAP global

- **Barras:** média do |SHAP| por variável = importância global.
- **Beeswarm:** cada ponto é uma transação. Eixo horizontal = efeito na previsão (direita empurra para fraude); cor = valor da variável (vermelho = alto). Mostra o **sentido** do efeito, não só a magnitude.

In [ ]:
explicador = shap.TreeExplainer(xgb)
shap_vals = explicador(amostra)  # valores em log-odds

shap.plots.bar(shap_vals, max_display=15, show=False)
plt.tight_layout()
plt.savefig("shap_importancia.png", dpi=120)
plt.show()

shap.plots.beeswarm(shap_vals, max_display=15, show=False)
plt.tight_layout()
plt.savefig("shap_beeswarm.png", dpi=120)
plt.show()

### SHAP local: uma fraude detectada

Waterfall: parte do valor base (média do modelo) e soma a contribuição de cada variável até a previsão final. Vermelho empurra para fraude; azul, para normal.

In [ ]:
proba_fraudes = pd.Series(proba_val, index=X_val.index)[idx_fraude]
idx_alvo = proba_fraudes.idxmax()  # fraude real com maior probabilidade prevista
print(f"Transação {idx_alvo}: probabilidade de fraude = {proba_fraudes[idx_alvo]:.4f}")

posicao = list(amostra.index).index(idx_alvo)
shap.plots.waterfall(shap_vals[posicao], max_display=12, show=False)
plt.savefig("shap_waterfall.png", dpi=120, bbox_inches="tight")
plt.show()

### SHAP local: uma fraude **perdida** (falso negativo)

Explicar só casos fáceis engana. Compare com uma fraude real que o modelo deixou passar (probabilidade abaixo do limiar escolhido).

In [ ]:
perdidas = proba_fraudes[proba_fraudes < limiar]
print(f"Fraudes perdidas na validação com limiar {limiar:.3f}: {len(perdidas)}")

idx_perdida = perdidas.idxmax()  # a perdida mais "perto" de ser detectada
print(f"Transação {idx_perdida}: probabilidade de fraude = {perdidas[idx_perdida]:.4f}")

pos_perdida = list(amostra.index).index(idx_perdida)
shap.plots.waterfall(shap_vals[pos_perdida], max_display=12, show=False)
plt.savefig("shap_waterfall_perdida.png", dpi=120, bbox_inches="tight")
plt.show()

### Cuidados na leitura

- SHAP mostra correlação do modelo, não causa.
- Valores em log-odds: somados ao valor base, dão a previsão.
- `Time` e `Amount` aparecem escalados (média 0, desvio 1) após o `StandardScaler`.

### Conclusões finais

**Resultados**

- **Desbalanceamento:** 0,17% de fraude (284 no treino, 94 na validação, 95 no teste). Um modelo que prevê sempre "normal" acerta ~99,8% e tem recall 0, então acurácia não serve; usamos recall, precisão e PR-AUC.
- **Escolha do modelo (validação):** o XGBoost com `scale_pos_weight` teve o melhor PR-AUC (0,883) e o melhor equilíbrio (recall 0,851, precisão 0,889, F1 0,870). `class_weight`, SMOTE e undersampling na regressão logística elevam o recall (0,89 a 0,90), mas a precisão cai para 0,05 a 0,42, o que bloquearia muitos clientes legítimos.
- **Teste (limiar 0,983, escolhido na validação):** recall 0,726, precisão 0,986, F1 0,836, PR-AUC 0,825, ROC-AUC 0,962. Foram 69 fraudes pegas, 26 perdidas e 1 alarme falso.
- **Limiar:** maximizar o F1 na validação levou a um limiar muito alto (0,983), que favorece a precisão e sacrifica recall (0,851 com limiar 0,5 na validação contra 0,726 no teste). Com ~94 fraudes, esse ponto é instável. Em produção o limiar deve vir do custo de negócio (por exemplo, "maior limiar com recall >= 90%").
- **O que o modelo usa:** `V14` domina (importância nativa 0,52; na fraude de maior probabilidade, +3,9 log-odds), seguida de `V4`, `V8`, `V12` e `V10`. Como `V1`-`V28` vêm de PCA, não dá para traduzir isso em regra de negócio.

**Limitações**

- Poucas fraudes: a diferença de 0,01 a 0,03 entre modelos e cada ponto de recall (1 fraude ~ 1 ponto) são estimativas ruidosas. Uma única divisão não basta; validação cruzada estratificada daria intervalos de confiança.
- Treino e teste vêm do mesmo período de ~2 dias, sem divisão temporal. Em produção a fraude muda com o tempo (drift) e o desempenho tende a ser pior.

**Melhorias a testar**

1. Escolher o limiar por custo/recall mínimo em vez do F1 máximo.
2. Validação cruzada estratificada e ajuste de hiperparâmetros do XGBoost (`max_depth`, `n_estimators`, `learning_rate`).
3. Engenharia de `Time` (hora do dia) e `log1p(Amount)`, decididos na exploração.
4. Divisão temporal (treinar no passado, testar no futuro) para medir drift.
5. Calibrar as probabilidades e monitorar recall e precisão após a implantação.

**Revisão do projeto (código e estrutura)**

O notebook está correto e sem duplicação: um fluxo único, o `modelo_final` e o `limiar` são reaproveitados entre as etapas. Nos scripts `.py` há problemas:

- A definição do pipeline XGBoost e o cálculo de `peso_fraude` estão copiados em `etapa3_treinar.py`, `etapa4_avaliar.py` e `etapa5_explicar.py`. Mudar um hiperparâmetro exige editar três arquivos, e eles podem divergir.
- A `URL` e o `read_csv` aparecem em `carregar_dados.py` e em `etapa1_explorar.py`; `drop_duplicates()` aparece em `etapa1_explorar.py` e `etapa2_preparar.py`.
- Cada importação refaz trabalho pesado: `etapa2_preparar.py` baixa o CSV e imprime ao ser importado, e `etapa4`/`etapa5` retreinam o modelo do zero. `etapa3_treinar.py` treina todos os candidatos ao ser importado.
- A pasta não tem `requirements.txt`, `README` nem `.gitignore`; há `__pycache__` e PNGs gerados junto do código.
- Os três scripts acima ficam redundantes com o notebook.

Estrutura sugerida: um módulo `modelo.py` (`construir_modelo(y_train)`) compartilhado pelas etapas 3 a 5, a limpeza de duplicatas apenas em `carregar_dados.py`, scripts com `if __name__ == "__main__":`, e `requirements.txt` + `.gitignore`. O notebook pode continuar como entrega principal.
